# 5.5 · 支持向量机 / Support Vector Machine (SVM)

> **课程定位 / Where this fits**
> 逻辑回归找**一条**能分开的边界, 但能分开的边界有无数条——哪条最好? SVM 答: **间隔(margin)最大**那条, 最稳健。再加**核技巧(kernel trick)**画非线性边界。SVR(4.9)是它的回归版。深度学习兴起前, SVM 是分类的王者。
> SVM picks the maximum-margin boundary, and the kernel trick lets it draw nonlinear ones. The pre-deep-learning champion.

> 💡 **面试相关 / Interview-relevant**
> - "什么是间隔 / 为什么最大间隔泛化好" ★★★★★
> - "支持向量是什么" ★★★★★
> - "核技巧的本质 / 为什么不显式升维" ★★★★★
> - "C 和 gamma 各控制什么" ★★★★★
> - "hinge loss vs 逻辑损失" ★★★★
> - "SVM 为什么要缩放" ★★★★

---

## 学习目标 / Learning Objectives
1. 最大间隔的几何 + 支持向量。
2. 软间隔与 **C**(正则强度)。
3. **hinge loss** 视角(对比逻辑损失)。
4. **核技巧**: RBF/多项式, 隐式升维。
5. **C 与 gamma** 调参 + 缩放必要性。

## 目录 / TOC
1. [最大间隔与支持向量 ⭐](#1)
2. [软间隔与 C ⭐](#2)
3. [hinge loss ⭐](#3)
4. [核技巧 ⭐](#4)
5. [🔢 数据 Digits + C/gamma 调参](#5)
6. [缩放与多分类](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 最大间隔与支持向量 ⭐ / Max Margin & Support Vectors

线性可分时, 能完美分开的超平面 $\mathbf{w}^\top\mathbf{x}+b=0$ 有无数个。SVM 选**离最近样本最远**的那个——**间隔(margin)最大化**。

间隔 = 边界到最近点的距离 $=\frac{1}{\|\mathbf{w}\|}$(每侧)。最大化间隔 ⟺ 最小化 $\|\mathbf{w}\|^2$, 约束所有点分类正确且在间隔外:
$$\min_{\mathbf{w},b}\tfrac12\|\mathbf{w}\|^2 \quad \text{s.t.}\quad y_i(\mathbf{w}^\top\mathbf{x}_i+b)\ge 1$$

**支持向量** = 恰好落在间隔边界上($y_i(\mathbf{w}^\top\mathbf{x}_i+b)=1$)的点。**只有它们决定边界**, 其余点移动不影响——这是 SVM 的稀疏性与稳健性来源。

**为什么大间隔泛化好**: 间隔大 = 边界对扰动不敏感 = 等价于一种复杂度控制(对应 $\|\mathbf{w}\|^2$ 正则)。


<a id="2"></a>
## 2. 软间隔与 C ⭐ / Soft Margin & C

真实数据常**线性不可分**或有噪声。软间隔引入松弛变量 $\xi_i\ge0$ 允许违反:
$$\min \tfrac12\|\mathbf{w}\|^2 + C\sum_i \xi_i \quad \text{s.t.}\quad y_i(\mathbf{w}^\top\mathbf{x}_i+b)\ge 1-\xi_i$$

**C 是正则旋钮**(与 4.4 岭回归的 $\lambda$ 反向):
- **C 大**: 重罚违反 → 间隔窄、努力分对每点 → **低偏差高方差**(过拟合)。
- **C 小**: 容忍违反 → 间隔宽、更平滑 → **高偏差低方差**。

C ≈ $1/\lambda$。这是 SVM 最重要的超参之一。


<a id="3"></a>
## 3. hinge loss ⭐ / Hinge Loss

软间隔可写成"损失 + 正则"形式(像 4.x 所有模型):
$$\min_{\mathbf{w}} \;\underbrace{\sum_i \max(0,\,1 - y_i\,f(\mathbf{x}_i))}_{\text{hinge loss}} + \frac{1}{2C}\|\mathbf{w}\|^2$$

**hinge loss** $\max(0, 1-yf)$: 分对且在间隔外($yf\ge1$)→ 损失 0; 否则线性惩罚。对比逻辑损失: hinge 在 $yf>1$ 后**完全为 0**(产生稀疏支持向量), 逻辑损失永远 $>0$(所有点都参与)。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

z = np.linspace(-2, 3, 200)   # z = y*f(x)
hinge = np.maximum(0, 1 - z)
logistic = np.log2(1 + np.exp(-z))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(z, hinge, lw=2, label="hinge (SVM): max(0,1-z)")
ax.plot(z, logistic, lw=2, label="logistic: log(1+e⁻ᶻ)")
ax.axvline(1, color="gray", ls="--", label="间隔边界 z=1")
ax.set_xlabel("z = y·f(x) (正确性·置信度)"); ax.set_ylabel("loss"); ax.legend()
ax.set_title("hinge 在 z≥1 后归零 → 稀疏支持向量; 逻辑损失永远>0")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. 核技巧 ⭐ / The Kernel Trick

线性不可分的数据, 升到高维常常可分(如同心圆 → 加 $r^2$ 维就线性可分)。但显式升维计算昂贵。

**核技巧的精髓**: SVM 的对偶形式里, 数据只以**内积** $\mathbf{x}_i^\top\mathbf{x}_j$ 出现。用**核函数** $K(\mathbf{x}_i,\mathbf{x}_j)=\phi(\mathbf{x}_i)^\top\phi(\mathbf{x}_j)$ 直接算高维内积, **无需显式计算 $\phi$**。常用核:
- **线性**: $\mathbf{x}_i^\top\mathbf{x}_j$
- **多项式**: $(\gamma\,\mathbf{x}_i^\top\mathbf{x}_j + r)^p$
- **RBF(高斯)**: $\exp(-\gamma\|\mathbf{x}_i-\mathbf{x}_j\|^2)$ —— 对应**无限维**空间, 最常用

**gamma($\gamma$)**(RBF): 控制单个样本影响范围。**gamma 大** → 影响范围小、边界弯曲 → 过拟合; **gamma 小** → 影响范围大、边界平滑。


In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_circles
Xc, yc = make_circles(200, factor=0.4, noise=0.1, random_state=0)  # 线性不可分

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
xx, yy = np.meshgrid(np.linspace(-1.5,1.5,300), np.linspace(-1.5,1.5,300))
for ax, kern in zip(axes, ["linear", "rbf"]):
    m = SVC(kernel=kern, C=1, gamma="scale").fit(Xc, yc)
    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(Xc[:,0], Xc[:,1], c=yc, cmap="coolwarm", edgecolor="k", s=25)
    ax.set_title(f"kernel={kern}: 准确率 {m.score(Xc,yc):.2f}")
plt.tight_layout(); plt.show()
print("同心圆: 线性核无能为力, RBF 核轻松画圆形边界 (隐式升维)")


<a id="5"></a>
## 5. 数据 Digits + C/gamma 调参 / Digits & Tuning

**Digits**: sklearn 内置手写数字(MNIST 的迷你版), 1797 张 8×8 灰度图, 0–9 十类。SVM 在这种图像分类上经典强。


In [ ]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC

digits = load_digits()
print(f"Digits: {digits.data.shape}, 10 类 0-9")
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img, lab in zip(axes, digits.images, digits.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(str(lab)); ax.axis("off")
plt.suptitle("Digits 样例 (8×8)"); plt.tight_layout(); plt.show()

X_tr, X_te, y_tr, y_te = train_test_split(digits.data, digits.target, test_size=0.3,
                                          stratify=digits.target, random_state=0)
sc = StandardScaler().fit(X_tr)
Xtr, Xte = sc.transform(X_tr), sc.transform(X_te)


In [ ]:
# C/gamma 网格搜索 / grid search
grid = GridSearchCV(SVC(kernel="rbf"),
                    {"C": [0.1, 1, 10], "gamma": [0.001, 0.01, 0.1]},
                    cv=3, n_jobs=-1).fit(Xtr, y_tr)
print(f"最佳参数: {grid.best_params_}")
print(f"test 准确率: {grid.score(Xte, y_te):.3f}")
print(f"支持向量数: {grid.best_estimator_.n_support_.sum()} / {len(Xtr)} 训练样本")
print("只有部分样本成为支持向量 → SVM 的稀疏性")


<a id="6"></a>
## 6. 缩放与多分类 / Scaling & Multiclass

- **必须缩放**: RBF 核含 $\|\mathbf{x}_i-\mathbf{x}_j\|^2$, 和 KNN 一样对量纲敏感(3.4)。
- **多分类**: SVM 本质二分类, sklearn 的 `SVC` 用 **OvO**(one-vs-one, $\binom{K}{2}$ 个分类器), `LinearSVC` 用 OvR。
- **缺点**: 大数据慢(训练约 $O(n^2)\sim O(n^3)$), 不直接给概率(需额外校准, 5.15)。


In [ ]:
# 缩放对 SVM 的影响 / scaling impact
# 注: Digits 像素本就在同一 0-16 量纲, 缩放帮助不大; 故意把一个特征放大暴露问题
Xb_tr = X_tr.copy(); Xb_tr[:, 0] *= 1000
Xb_te = X_te.copy(); Xb_te[:, 0] *= 1000
unscaled = SVC(kernel="rbf", gamma="scale").fit(Xb_tr, y_tr).score(Xb_te, y_te)
scaled = SVC(kernel="rbf", gamma="scale").fit(
    StandardScaler().fit_transform(Xb_tr), y_tr).score(
    StandardScaler().fit(Xb_tr).transform(Xb_te), y_te)
print(f"某特征×1000 不缩放: {unscaled:.3f}   标准化后: {scaled:.3f}")
print("(同量纲特征缩放影响小, 但量纲不一时不缩放会被大特征绑架, 同 KNN)")

# 多分类: SVC 内部训练 C(10,2)=45 个 OvO 二分类器
svc = SVC(decision_function_shape="ovo").fit(Xtr, y_tr)
print(f"\nSVC 用 OvO; decision_function(ovo) 形状 = {svc.decision_function(Xte).shape}")
print("(45 = C(10,2), 10 类两两配对; 默认 decision_function_shape='ovr' 会聚合成 10 列)")


<a id="7"></a>
## 7. 小结 / Summary

```
SVM: 最大间隔 = 1/‖w‖ 最大化; 只有支持向量(间隔边界上的点)决定边界
软间隔 + C: C 大→窄间隔/过拟合, C 小→宽间隔/欠拟合 (C≈1/λ)
hinge loss: max(0,1-yf), z≥1 归零 → 稀疏支持向量 (vs 逻辑损失永远>0)
核技巧: 数据只以内积出现 → 用 K(xᵢ,xⱼ) 算高维内积, 不显式升维
  RBF: exp(-γ‖xᵢ-xⱼ‖²), γ大→边界弯曲/过拟合
必须缩放; 多分类 SVC=OvO; 大数据慢, 不直接给概率
```

### 💡 面试速查
1. **最大间隔**最稳健; **支持向量**是唯一决定边界的点
2. **C** 控正则(大=过拟合); **gamma** 控 RBF 影响范围(大=过拟合)
3. **核技巧**: 用核函数算高维内积, 避免显式升维; RBF 对应无限维
4. **hinge loss** 产生稀疏支持向量; **必须缩放**
5. 大数据慢、不直接给概率是 SVM 的主要缺点

### 下一节
**5.6 决策树**——SVM 是平滑边界, 决策树是**轴对齐的阶梯**边界, 可解释性极强, 也是随机森林/GBDT 的基石。
